In [1]:
"""
Signature Verification Training with ArcFace + Contrastive Loss
================================================================
"""

import os
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

# ── Imports from your project files ──────────────────────────────────────────
from model import model_final
from dataset import (train_dataloader, test_dataloader, val_dataloader,
                     train_pairs, val_pairs, test_pairs, genuine_by_author)

# ─────────────────────────────────────────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────────
# UNWRAP torch.compile  →  raw nn.Module  (avoids compile+fork deadlock)
# ─────────────────────────────────────────────────────────────────────────────
def unwrap_model(m):
    """Strip torch.compile / DataParallel wrappers."""
    if hasattr(m, '_orig_mod'):       # torch.compile wrapper
        return m._orig_mod
    if hasattr(m, 'module'):          # DataParallel / DistributedDataParallel
        return m.module
    return m

raw_model = unwrap_model(model_final).to(device)


# ─────────────────────────────────────────────────────────────────────────────
# SIMPLE FLAT DATASET  (no nested Dataset, no image_cache, no workers)
# ─────────────────────────────────────────────────────────────────────────────
class FlatPairDataset(Dataset):
    """
    Dead-simple dataset: reads images from disk, applies transform.
    - No image_cache (avoids pickle issues across worker forks)
    - No nested Dataset objects
    - Works perfectly with num_workers=0
    """
    def __init__(self, pairs, labels, author2id, transform):
        self.pairs     = pairs
        self.labels    = labels
        self.author2id = author2id
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    @staticmethod
    def _read(path):
        return Image.open(path).convert('L')

    @staticmethod
    def _author(path):
        try:
            return os.path.basename(path).split('_')[1]
        except IndexError:
            return "unknown"

    def __getitem__(self, idx):
        p1, p2  = self.pairs[idx]
        label   = self.labels[idx]

        img1 = self.transform(self._read(p1))
        img2 = self.transform(self._read(p2))

        a1 = self.author2id.get(self._author(p1), 0)
        a2 = self.author2id.get(self._author(p2), 0)

        return (img1, img2,
                torch.tensor(label, dtype=torch.float32),
                torch.tensor(a1,    dtype=torch.long),
                torch.tensor(a2,    dtype=torch.long))


# ─────────────────────────────────────────────────────────────────────────────
# ARCFACE LOSS
# ─────────────────────────────────────────────────────────────────────────────
class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, num_classes, s=32.0, m=0.50):
        super().__init__()
        self.s = s; self.m = m
        self.weight  = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m   = math.cos(m)
        self.sin_m   = math.sin(m)
        self.th      = math.cos(math.pi - m)
        self.mm      = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        W      = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings, W)
        sine   = torch.sqrt((1.0 - cosine.pow(2)).clamp(1e-9, 1.0))
        phi    = cosine * self.cos_m - sine * self.sin_m
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        logits = (one_hot * phi + (1.0 - one_hot) * cosine) * self.s
        return F.cross_entropy(logits, labels)


# ─────────────────────────────────────────────────────────────────────────────
# CONTRASTIVE LOSS
# ─────────────────────────────────────────────────────────────────────────────
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, labels):
        d   = F.pairwise_distance(emb1, emb2, p=2)
        pos = labels       * d.pow(2)
        neg = (1 - labels) * F.relu(self.margin - d).pow(2)
        return (pos + neg).mean()


# ─────────────────────────────────────────────────────────────────────────────
# COMBINED LOSS
# ─────────────────────────────────────────────────────────────────────────────
class SignatureVerificationLoss(nn.Module):
    def __init__(self, num_classes, emb_dim=256, s=32.0, m=0.50,
                 margin=1.0, lambda_arc=0.6, lambda_con=0.4):
        super().__init__()
        self.arcface     = ArcFaceLoss(emb_dim, num_classes, s=s, m=m)
        self.contrastive = ContrastiveLoss(margin=margin)
        self.la = lambda_arc
        self.lc = lambda_con

    def forward(self, emb1, emb2, pair_labels, auth1, auth2):
        arc = (self.arcface(emb1, auth1) + self.arcface(emb2, auth2)) / 2.0
        con = self.contrastive(emb1, emb2, pair_labels)
        total = self.la * arc + self.lc * con
        return total, arc.item(), con.item()


# ─────────────────────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def compute_distances(model, loader):
    model.eval()
    dists, labs = [], []
    for img1, img2, labels, _, _ in loader:
        img1, img2 = img1.to(device), img2.to(device)
        e1, e2 = model(img1, img2)
        dists.append(F.pairwise_distance(e1, e2).cpu().numpy())
        labs.append(labels.numpy())
    return np.concatenate(dists), np.concatenate(labs)


def best_threshold(distances, labels):
    best_acc, best_t = 0.0, 0.5
    for t in np.linspace(distances.min(), distances.max(), 200):
        acc = accuracy_score(labels, (distances < t).astype(int))
        if acc > best_acc:
            best_acc, best_t = acc, t
    return best_t, best_acc


def evaluate(model, loader, split="Val"):
    dists, labels = compute_distances(model, loader)
    t, acc        = best_threshold(dists, labels)
    try:
        auc = roc_auc_score(labels, -dists)
    except ValueError:
        auc = 0.5
    preds   = (dists < t).astype(int)
    genuine = labels == 1;  forged = labels == 0
    far = float(np.mean(preds[forged]  == 1)) if forged.any()  else 0.0
    frr = float(np.mean(preds[genuine] == 0)) if genuine.any() else 0.0
    print(f"  [{split}] Acc={acc:.4f}  AUC={auc:.4f}  FAR={far:.4f}  FRR={frr:.4f}  Thresh={t:.4f}")
    return {"acc": acc, "auc": auc, "far": far, "frr": frr, "threshold": t}


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN / VAL EPOCH
# ─────────────────────────────────────────────────────────────────────────────
def train_epoch(model, criterion, optimizer, scheduler, loader, epoch):
    model.train()
    tot = tot_arc = tot_con = 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch:03d} [Train]", leave=False)

    for img1, img2, pair_labels, auth1, auth2 in pbar:
        img1, img2       = img1.to(device),       img2.to(device)
        pair_labels      = pair_labels.to(device)
        auth1, auth2     = auth1.to(device),       auth2.to(device)

        optimizer.zero_grad()
        e1, e2           = model(img1, img2)
        loss, arc, con   = criterion(e1, e2, pair_labels, auth1, auth2)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()

        tot += loss.item(); tot_arc += arc; tot_con += con
        pbar.set_postfix(loss=f"{loss.item():.4f}", arc=f"{arc:.4f}", con=f"{con:.4f}")

    n = len(loader)
    print(f"  [Train] Loss={tot/n:.4f}  ArcFace={tot_arc/n:.4f}  Contrastive={tot_con/n:.4f}")
    return tot / n


@torch.no_grad()
def val_epoch(model, criterion, loader):
    model.eval()
    tot = 0.0
    for img1, img2, pair_labels, auth1, auth2 in loader:
        img1, img2   = img1.to(device), img2.to(device)
        pair_labels  = pair_labels.to(device)
        auth1, auth2 = auth1.to(device), auth2.to(device)
        e1, e2       = model(img1, img2)
        loss, _, _   = criterion(e1, e2, pair_labels, auth1, auth2)
        tot += loss.item()
    avg = tot / len(loader)
    print(f"  [Val  ] Loss={avg:.4f}")
    return avg


# ─────────────────────────────────────────────────────────────────────────────
# MAIN TRAINING FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def train(
    model,
    train_dataloader, val_dataloader, test_dataloader,
    train_pairs, val_pairs, test_pairs,
    genuine_by_author,
    num_epochs        = 40,
    lr                = 3e-4,
    weight_decay      = 1e-4,
    arcface_margin    = 0.50,
    arcface_scale     = 32.0,
    contrastive_margin= 1.0,
    lambda_arc        = 0.6,
    lambda_con        = 0.4,
    save_dir          = "./checkpoints",
    patience          = 10,
):
    os.makedirs(save_dir, exist_ok=True)

    # ── Author → integer id ──────────────────────────────────────────────────
    author2id   = {a: i for i, a in enumerate(sorted(genuine_by_author.keys()))}
    num_classes = len(author2id)
    print(f"Number of signer classes (ArcFace): {num_classes}")

    # ── Kill ALL original persistent worker pools ────────────────────────────
    for old_loader in [train_dataloader, val_dataloader, test_dataloader]:
        try:
            old_loader._iterator = None
        except Exception:
            pass
    # Give OS time to reap worker processes
    import time; time.sleep(1)

    # ── Grab transforms from original datasets ───────────────────────────────
    train_tf = train_dataloader.dataset.transform
    val_tf   = val_dataloader.dataset.transform
    test_tf  = test_dataloader.dataset.transform

    # ── Build flat datasets (num_workers=0 → no forking, no deadlock) ────────
    def make_loader(pairs, labels, transform, batch_size, shuffle):
        ds = FlatPairDataset(pairs, labels, author2id, transform)
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          num_workers=0, pin_memory=torch.cuda.is_available())

    train_labels = train_dataloader.dataset.labels
    val_labels   = val_dataloader.dataset.labels
    test_labels  = test_dataloader.dataset.labels

    train_loader = make_loader(train_pairs, train_labels, train_tf, 64,  True)
    val_loader   = make_loader(val_pairs,   val_labels,   val_tf,  128, False)
    test_loader  = make_loader(test_pairs,  test_labels,  test_tf, 128, False)

    print(f"  Train: {len(train_loader)} batches  |  "
          f"Val: {len(val_loader)} batches  |  "
          f"Test: {len(test_loader)} batches")

    # ── Loss ─────────────────────────────────────────────────────────────────
    criterion = SignatureVerificationLoss(
        num_classes, emb_dim=256, s=arcface_scale, m=arcface_margin,
        margin=contrastive_margin, lambda_arc=lambda_arc, lambda_con=lambda_con,
    ).to(device)

    # ── Optimizer ────────────────────────────────────────────────────────────
    optimizer = AdamW([
        {"params": model.parameters(),              "lr": lr,        "weight_decay": weight_decay},
        {"params": criterion.arcface.parameters(),  "lr": lr * 0.1,  "weight_decay": weight_decay},
    ])

    # ── Scheduler ────────────────────────────────────────────────────────────
    scheduler = OneCycleLR(
        optimizer,
        max_lr          = [lr, lr * 0.1],
        total_steps     = num_epochs * len(train_loader),
        pct_start       = 0.1,
        anneal_strategy = "cos",
        div_factor      = 25,
        final_div_factor= 1e4,
    )

    # ── Training loop ─────────────────────────────────────────────────────────
    best_auc, best_epoch, no_improve = 0.0, 0, 0
    history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_acc": []}

    print("\n" + "="*65)
    print("TRAINING  —  ArcFace + Contrastive Loss")
    print("="*65)

    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs}")
        tl = train_epoch(model, criterion, optimizer, scheduler, train_loader, epoch)
        vl = val_epoch(model, criterion, val_loader)
        vm = evaluate(model, val_loader, "Val")

        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["val_auc"].append(vm["auc"])
        history["val_acc"].append(vm["acc"])

        if vm["auc"] > best_auc:
            best_auc, best_epoch, no_improve = vm["auc"], epoch, 0
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "arc_state": criterion.arcface.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "val_auc": best_auc, "threshold": vm["threshold"],
                "author2id": author2id,
            }, os.path.join(save_dir, "best_model.pt"))
            print(f"  ✓ Best saved  AUC={best_auc:.4f}")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  ⏹ Early stop at epoch {epoch}")
                break

    # ── Test ──────────────────────────────────────────────────────────────────
    print("\n" + "="*65 + "\nTEST EVALUATION\n" + "="*65)
    ckpt = torch.load(os.path.join(save_dir, "best_model.pt"), map_location=device)
    model.load_state_dict(ckpt["model_state"])
    tm = evaluate(model, test_loader, "Test")
    print(f"\nBest epoch={best_epoch}  Val AUC={best_auc:.4f}")
    print(f"Test → AUC={tm['auc']:.4f}  Acc={tm['acc']:.4f}  FAR={tm['far']:.4f}  FRR={tm['frr']:.4f}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"],   label="Val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
    axes[1].plot(history["val_auc"])
    axes[1].set_title("Val AUC"); axes[1].set_xlabel("Epoch")
    axes[2].plot(history["val_acc"])
    axes[2].set_title("Val Accuracy"); axes[2].set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150)
    plt.close()

    return model, history, tm


# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE HELPER
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def verify_pair(model, img1_tensor, img2_tensor, threshold=0.5):
    model.eval()
    e1, e2 = model(img1_tensor.to(device), img2_tensor.to(device))
    dist   = F.pairwise_distance(e1, e2).item()
    return {
        "distance":   dist,
        "threshold":  threshold,
        "is_genuine": dist < threshold,
        "confidence": float(np.clip(1.0 - dist / (threshold * 2), 0, 1)),
        "verdict":    "GENUINE" if dist < threshold else "FORGED",
    }


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    trained_model, history, test_metrics = train(
        model              = raw_model,
        train_dataloader   = train_dataloader,
        val_dataloader     = val_dataloader,
        test_dataloader    = test_dataloader,
        train_pairs        = train_pairs,
        val_pairs          = val_pairs,
        test_pairs         = test_pairs,
        genuine_by_author  = genuine_by_author,
        num_epochs         = 40,
        lr                 = 3e-4,
        weight_decay       = 1e-4,
        arcface_margin     = 0.50,
        arcface_scale      = 32.0,
        contrastive_margin = 1.0,
        lambda_arc         = 0.6,
        lambda_con         = 0.4,
        save_dir           = "./checkpoints",
        patience           = 10,
    )

Genuine authors: 268, dict_keys(['001', '002', '003', '004', '006', '009', '012', '014', '015', '016', '021', '022', '023', '024', '025', '026', '027', '028', '029', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061', '068', '069', '070', '071', '072', '073', '074', '075', '076', '077', '084', '085', '086', '087', '088', '089', '090', '091', '092', '093', '100', '101', '102', '103', '104', '105', '118', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '171', '172', '173', '174', '230', '231', '233', '234', '235', '238', '240', '241', '243', '246', '249', '250', '251', '315', '318', '320', '328', '333', '341', '

  [Train] Loss=12.5858  ArcFace=20.7788  Contrastive=0.2963
  [Val  ] Loss=13.5866
  [Val] Acc=0.8406  AUC=0.9145  FAR=0.2073  FRR=0.1115  Thresh=0.4037
  ✓ Best saved  AUC=0.9145

Epoch 2/40


  [Train] Loss=10.1529  ArcFace=16.6715  Contrastive=0.3750
  [Val  ] Loss=14.6658
  [Val] Acc=0.8604  AUC=0.9441  FAR=0.1627  FRR=0.1165  Thresh=0.7113
  ✓ Best saved  AUC=0.9441

Epoch 3/40


  [Train] Loss=5.4228  ArcFace=8.7843  Contrastive=0.3806
  [Val  ] Loss=13.7465
  [Val] Acc=0.8556  AUC=0.9392  FAR=0.1670  FRR=0.1219  Thresh=0.8536

Epoch 4/40


  [Train] Loss=2.6878  ArcFace=4.2703  Contrastive=0.3140
  [Val  ] Loss=10.7854
  [Val] Acc=0.8614  AUC=0.9433  FAR=0.1465  FRR=0.1307  Thresh=0.8400

Epoch 5/40


...

  [Train] Loss=0.1056  ArcFace=0.0375  Contrastive=0.2078
  [Val  ] Loss=0.7679
  [Val] Acc=0.9538  AUC=0.9890  FAR=0.0632  FRR=0.0291  Thresh=0.3692

Epoch 26/40


  [Train] Loss=0.1041  ArcFace=0.0359  Contrastive=0.2064
  [Val  ] Loss=0.7442
  [Val] Acc=0.9587  AUC=0.9920  FAR=0.0507  FRR=0.0319  Thresh=0.3729
  ✓ Best saved  AUC=0.9920

Epoch 27/40


  [Train] Loss=0.1013  ArcFace=0.0320  Contrastive=0.2053
  [Val  ] Loss=0.7675
  [Val] Acc=0.9606  AUC=0.9917  FAR=0.0658  FRR=0.0129  Thresh=0.3954

Epoch 28/40


  [Train] Loss=0.1001  ArcFace=0.0303  Contrastive=0.2047
  [Val  ] Loss=0.7683
  [Val] Acc=0.9569  AUC=0.9903  FAR=0.0656  FRR=0.0207  Thresh=0.3914

Epoch 29/40


KeyboardInterrupt: 